# Housing area groups

Each row is a California census block group, not a single house. K-Means forms six groups from `Latitude`, `Longitude`, and `MedInc`. A later classifier tries to reproduce those group ids.

The reproducible run is `python src/app.py` from the repository root. This notebook calls the same functions for loading, clustering, profiles, and figures. House value is not a target.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from kmeans_housing import (
    DATASET_URL,
    FEATURES,
    RANDOM_STATE,
    TEST_SIZE,
    artifact_paths,
    assign_clusters,
    build_cluster_profiles,
    build_kmeans_pipeline,
    describe_clusters,
    load_housing_data,
    save_cluster_map,
    save_income_plot,
    split_features,
    summarize_features,
    validate_and_select_features,
)

paths = artifact_paths(ROOT)
print(DATASET_URL)
print("Feature order:", FEATURES)

## Load and check the three features

The local file is the root `housing.csv`. Missing, non-numeric, or infinite values in the three required columns stop the workflow. Duplicate three-feature rows are counted and kept. Coordinates outside the physical latitude and longitude ranges are flagged and kept.

In [ ]:
raw = load_housing_data(paths["data"])
features = validate_and_select_features(raw, FEATURES)
diagnostics = summarize_features(features)
print("Source columns:", list(raw.columns))
print("Model columns:", list(features.columns))
print("Rows:", diagnostics["row_count"])
print("Duplicate three-feature rows kept:", diagnostics["duplicate_feature_rows"])
print("Coordinate flags:", diagnostics["coordinate_flags"])
features.describe()

## Fit six clusters on the training rows

The split happens before scaling. The scaler and K-Means fit on the training rows only, with seed 42. Test labels come from that same fitted pipeline.

In [ ]:
x_train, x_test = split_features(features, TEST_SIZE, RANDOM_STATE)
pipeline = build_kmeans_pipeline(RANDOM_STATE)
pipeline.fit(x_train)
train_labels, test_labels = assign_clusters(pipeline, x_train, x_test)
profiles = build_cluster_profiles(x_train, train_labels, pipeline)
print(f"Train rows: {len(x_train)}; test rows: {len(x_test)}")
for statement in describe_clusters(x_train, profiles):
    print("-", statement)
profiles

## Maps and income

Color is the cluster id and stays the same across figures. Circles are training rows. X markers are test predictions. Income is shown separately because a map of longitude and latitude cannot show the third feature.

In [ ]:
save_cluster_map(x_train, train_labels, paths["train_plot"])
save_cluster_map(
    x_train,
    train_labels,
    paths["train_test_plot"],
    x_test=x_test,
    test_labels=test_labels,
)
save_income_plot(x_train, train_labels, paths["income_plot"])

from IPython.display import Image, display

for figure in (paths["train_plot"], paths["train_test_plot"], paths["income_plot"]):
    display(Image(filename=str(figure)))

## Supervised fidelity

Classifier comparison and the holdout score are produced by `python src/app.py` and stored in `models/model_metadata.json`. The score is agreement with K-Means, not accuracy against a human label. On the completed run, random forest had the highest training macro F1, and its holdout agreement was about 0.995 accuracy and 0.993 macro F1.

In [ ]:
import json

metadata_path = paths["metadata"]
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text())
    selection = metadata["supervised_selection"]
    print("Selected:", selection["winner"])
    print(pd.DataFrame(selection["candidates"]).T)
    fidelity = metadata["holdout_fidelity"]
    print(
        "Holdout accuracy:",
        round(fidelity["accuracy"], 4),
        "macro F1:",
        round(fidelity["macro_f1"], 4),
    )
    display(Image(filename=str(paths["confusion_plot"])))
else:
    print("Run python src/app.py to compare classifiers and write the metadata.")